# Extract All Data Needed for Section 4.5
Run all cells, then screenshot the outputs for Claude.

In [1]:
import os, numpy as np, pandas as pd
from iohblade.loggers import ExperimentLogger
from iohblade.behaviour_metrics import compute_behavior_metrics
import iohinspector

EXPERIMENT_DIRS = [
    '../results/EoH',
    '../results/CROSSOVER-ABLATION/baseline-LLAMEA',
    '../results/GA-LLAMEA-8-INIT-100',
]
IOH_ALL_DIR = '../results/ioh-all'

ALL_FEATS = [
    'avg_nearest_neighbor_distance','dispersion','avg_exploration_pct',
    'avg_distance_to_best','intensification_ratio','avg_exploitation_pct',
    'average_convergence_rate','avg_improvement','success_rate',
    'longest_no_improvement_streak','last_improvement_fraction',
]

STN_FEATS = [
    'avg_exploration_pct','average_convergence_rate','avg_improvement',
    'success_rate','longest_no_improvement_streak',
]

NICE = {
    'avg_nearest_neighbor_distance':'NN-dist','dispersion':'Disp',
    'avg_exploration_pct':'Expl%','avg_distance_to_best':'Dist->best',
    'intensification_ratio':'Inten-ratio','avg_exploitation_pct':'Explt%',
    'average_convergence_rate':'Conv-rate','avg_improvement':'Delta fitness',
    'success_rate':'Success%','longest_no_improvement_streak':'No-imp streak',
    'last_improvement_fraction':'Last-imp frac',
}

print('Setup done.')

c:\Users\Kukoy\Documents\Experiment-GA\WORKING-GA-LLAMEA\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup done.


In [2]:
# Load all algorithms
exp_logger = ExperimentLogger(EXPERIMENT_DIRS[0], True)
for d in EXPERIMENT_DIRS[1:]:
    exp_logger.add_read_dir(d)
methods, _ = exp_logger.get_methods_problems()

all_algos = exp_logger.get_problem_data('MA_BBOB')
all_algos.replace([-np.inf], 0, inplace=True)
all_algos.fillna(0, inplace=True)
all_algos = all_algos[all_algos['code'].notna() & (all_algos['code'] != '')].copy()

# Load behaviour metrics from IOH trajectory data
behaviour_records = []
for method in sorted(methods):
    method_algos = all_algos[all_algos['method_name'] == method]
    method_dir = os.path.join(IOH_ALL_DIR, method)
    count = 0
    for _, row in method_algos.iterrows():
        algo_id = str(row['id'])
        out_dir = os.path.join(method_dir, algo_id)
        if not os.path.isdir(out_dir):
            continue
        try:
            m = iohinspector.DataManager()
            m.add_folder(out_dir)
            df_traj = m.load(monotonic=False, include_meta_data=True).to_pandas()
        except: continue
        if df_traj.empty: continue
        run_metrics = []
        for (inst, run), grp in df_traj.groupby(['instance', 'run_id']):
            grp = grp.sort_values('evaluations').reset_index(drop=True)
            if len(grp) < 10: continue
            try: m_dict = compute_behavior_metrics(grp)
            except: m_dict = {f: np.nan for f in ALL_FEATS}
            run_metrics.append(m_dict)
        if not run_metrics: continue
        agg = pd.DataFrame(run_metrics)[ALL_FEATS].median().to_dict()
        agg['algo_id'] = algo_id
        agg['method_name'] = method
        agg['aocc'] = float(row['fitness'])
        behaviour_records.append(agg)
        count += 1
    print(f'{method}: {count} algorithms with trajectory data')

df = pd.DataFrame(behaviour_records)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna(0, inplace=True)
df['aocc_norm'] = df.groupby('method_name')['aocc'].transform(
    lambda s: (s - s.min()) / (s.max() - s.min() + 1e-12)
)
print(f'\nTotal behaviour records: {len(df)}')

Baseline-LLaMEA: 472 algorithms with trajectory data
EoH: 482 algorithms with trajectory data
GA-LLAMEA-8-INIT-100: 488 algorithms with trajectory data

Total behaviour records: 1442


In [3]:
# === TABLE 1: Median behaviour metrics per method vs Q4 Gold Standard ===
q4_threshold = df['aocc_norm'].quantile(0.75)
gold = df[df['aocc_norm'] >= q4_threshold][ALL_FEATS].median()
meds = df.groupby('method_name')[ALL_FEATS].median()

tbl = meds.T.copy()
tbl['Q4 Gold Std'] = gold
tbl.index = [NICE.get(f, f) for f in tbl.index]
print('TABLE 1: Median behaviour metrics per method vs Q4 Gold Standard')
display(tbl.round(4))

TABLE 1: Median behaviour metrics per method vs Q4 Gold Standard


method_name,Baseline-LLaMEA,EoH,GA-LLAMEA-8-INIT-100,Q4 Gold Std
NN-dist,0.1827,0.1937,0.1358,0.1018
Disp,6.9467,7.0195,6.9970,7.4230
Expl%,9.7038,10.9453,6.6566,4.0268
Dist->best,0.6312,0.7277,0.4241,0.2441
Inten-ratio,0.8362,0.7761,0.8927,0.9395
Explt%,90.2962,89.0547,93.3434,95.9732
Conv-rate,0.9962,0.9962,0.9962,0.9962
Delta fitness,0.1681,0.1437,0.1242,0.1122
Success%,0.0089,0.0085,0.0091,0.0092
No-imp streak,1753.2500,1924.2500,2795.0000,4482.0000


In [4]:
# === TABLE 2: Mean +/- Std for all 11 features per method ===
rows = []
for feat in ALL_FEATS:
    row = {'Metric': NICE[feat]}
    for method in sorted(methods):
        sub = df[df['method_name'] == method][feat]
        row[method] = f'{sub.mean():.4f} +/- {sub.std():.4f}'
    rows.append(row)
tbl2 = pd.DataFrame(rows).set_index('Metric')
print('TABLE 2: Mean +/- Std per method (all 11 features)')
display(tbl2)

TABLE 2: Mean +/- Std per method (all 11 features)


,Baseline-LLaMEA,EoH,GA-LLAMEA-8-INIT-100
Metric,,,
NN-dist,1.2481 +/- 2.0364,0.6826 +/- 1.3052,0.8142 +/- 1.6164
Disp,7.0627 +/- 1.2045,6.9334 +/- 1.8305,6.8428 +/- 1.2758
Expl%,31.1901 +/- 37.2481,26.2481 +/- 31.1869,22.7940 +/- 32.9934
Dist->best,2.4177 +/- 3.1307,2.0958 +/- 2.7642,1.7714 +/- 2.8260
Inten-ratio,0.6095 +/- 0.3895,0.6093 +/- 0.3684,0.7163 +/- 0.3502
Explt%,68.8099 +/- 37.2481,73.7519 +/- 31.1869,77.2060 +/- 32.9934
Conv-rate,0.8513 +/- 0.3007,0.9476 +/- 0.1657,0.9189 +/- 0.2172
Delta fitness,1.0372 +/- 1.6494,0.5640 +/- 1.0953,0.6556 +/- 1.3208
Success%,0.0238 +/- 0.0338,0.0241 +/- 0.0638,0.0169 +/- 0.0223


In [5]:
# === TABLE 3: Pearson correlation of each feature with normalised AOCC ===
print('TABLE 3: Feature correlations with normalised AOCC')
corr_rows = []
for feat in ALL_FEATS:
    row = {'Metric': NICE[feat], 'Overall': round(df[feat].corr(df['aocc_norm']), 3)}
    for method in sorted(methods):
        sub = df[df['method_name'] == method]
        row[method] = round(sub[feat].corr(sub['aocc_norm']), 3)
    corr_rows.append(row)
display(pd.DataFrame(corr_rows).set_index('Metric'))

TABLE 3: Feature correlations with normalised AOCC


,Overall,Baseline-LLaMEA,EoH,GA-LLAMEA-8-INIT-100
Metric,,,,
NN-dist,-0.319,-0.262,-0.118,-0.523
Disp,0.031,0.048,0.053,0.019
Expl%,-0.460,-0.372,-0.351,-0.622
Dist->best,-0.452,-0.360,-0.347,-0.615
Inten-ratio,0.551,0.453,0.509,0.667
Explt%,0.460,0.372,0.351,0.622
Conv-rate,0.234,0.165,0.065,0.440
Delta fitness,-0.312,-0.253,-0.118,-0.513
Success%,-0.204,-0.129,-0.216,-0.379


In [6]:
# === TABLE 4: Q4 algorithm counts + median AOCC per method ===
print('TABLE 4: Q4 counts and median AOCC')
q4_mask = df['aocc_norm'] >= q4_threshold
summary_rows = []
for method in sorted(methods):
    sub = df[df['method_name'] == method]
    q4c = q4_mask[sub.index].sum()
    summary_rows.append({
        'Method': method,
        'N algos': len(sub),
        'Median AOCC': round(sub['aocc'].median(), 4),
        'Mean AOCC': round(sub['aocc'].mean(), 4),
        'Max AOCC': round(sub['aocc'].max(), 4),
        'Q4 count': q4c,
        'Q4 %': f"{100*q4c/len(sub):.1f}%",
    })
display(pd.DataFrame(summary_rows).set_index('Method'))

TABLE 4: Q4 counts and median AOCC


,N algos,Median AOCC,Mean AOCC,Max AOCC,Q4 count,Q4 %
Method,,,,,,
Baseline-LLaMEA,472,0.5646,0.5315,0.8566,99,21.0%
EoH,482,0.4936,0.5081,0.8373,85,17.6%
GA-LLAMEA-8-INIT-100,488,0.7376,0.6028,0.8640,177,36.3%


In [7]:
# === TABLE 5: Redundant feature pairs (|r| >= 0.7) ===
print('TABLE 5: Redundant feature pairs')
feat_corr = df[ALL_FEATS].corr()
fitness_corr = df[ALL_FEATS + ['aocc_norm']].corr()['aocc_norm'].drop('aocc_norm').abs()
pairs = []
for i, f1 in enumerate(ALL_FEATS):
    for j, f2 in enumerate(ALL_FEATS):
        if j <= i: continue
        r = feat_corr.loc[f1, f2]
        if abs(r) >= 0.7:
            keep = f1 if fitness_corr[f1] >= fitness_corr[f2] else f2
            drop = f2 if keep == f1 else f1
            pairs.append({'A': NICE[f1], 'B': NICE[f2], 'r': round(r,2),
                          'Keep': NICE[keep], 'Drop': NICE[drop]})
display(pd.DataFrame(pairs))

TABLE 5: Redundant feature pairs


,A,B,r,Keep,Drop
0,NN-dist,Expl%,0.88,Expl%,NN-dist
1,NN-dist,Dist->best,0.87,Dist->best,NN-dist
2,NN-dist,Inten-ratio,-0.73,Inten-ratio,NN-dist
3,NN-dist,Explt%,-0.88,Explt%,NN-dist
4,NN-dist,Conv-rate,-0.96,NN-dist,Conv-rate
5,NN-dist,Delta fitness,0.98,NN-dist,Delta fitness
6,Expl%,Dist->best,0.98,Expl%,Dist->best
7,Expl%,Inten-ratio,-0.92,Inten-ratio,Expl%
8,Expl%,Explt%,-1.00,Expl%,Explt%
9,Expl%,Conv-rate,-0.77,Expl%,Conv-rate


In [8]:
# === TABLE 6: STN feature medians per method (the 5 paper features) ===
print('TABLE 6: STN feature medians per method vs Q4 Gold Standard')
stn_meds = df.groupby('method_name')[STN_FEATS].median()
stn_tbl = stn_meds.T.copy()
stn_tbl['Q4 Gold Std'] = df[df['aocc_norm'] >= q4_threshold][STN_FEATS].median()
stn_tbl.index = [NICE[f] for f in stn_tbl.index]
display(stn_tbl.round(4))

print('\n=== Q4 threshold value:', round(q4_threshold, 4))

TABLE 6: STN feature medians per method vs Q4 Gold Standard


method_name,Baseline-LLaMEA,EoH,GA-LLAMEA-8-INIT-100,Q4 Gold Std
Expl%,9.7038,10.9453,6.6566,4.0268
Conv-rate,0.9962,0.9962,0.9962,0.9962
Delta fitness,0.1681,0.1437,0.1242,0.1122
Success%,0.0089,0.0085,0.0091,0.0092
No-imp streak,1753.2500,1924.2500,2795.0000,4482.0000



=== Q4 threshold value: 0.8802
